# Fase 3 — Notebook 04: Análise Fatorial e os Pesos do IVS

**Entrada:** `banco_de_dados/entrega_orientadora/Base_ELSI_70Municipios_Censo2022.db`,
tabela `setores_censitarios` — o SQLite versionado da entrega. Este notebook roda **sem**
os 2,4 GB de microdados do Censo.

**Objetivo:** estimar a estrutura latente dos componentes do IVS e definir os **pesos** do
índice. O produto são os pesos e a estrutura — **não** o IVS final, que é o Notebook 05,
depois da normalização municipal do Notebook 03.

**Referência metodológica:**
- MATOS, D. A. S.; RODRIGUES, E. C. *Análise fatorial*. Brasília: Enap, 2019. 74 p. — a
  referência principal. As páginas citadas ao longo do notebook são as da numeração
  impressa.
- FIGUEIREDO FILHO, D. B.; SILVA JÚNIOR, J. A. Visão além do alcance: uma introdução à
  análise fatorial. *Opinião Pública*, v. 16, n. 1, 2010 — a referência anterior, que a
  Enap contradiz em três pontos (rotação, técnica de extração e regra de comunalidade).
- `docs/Analise_Fatorial_Figueiredo2010_e_o_Projeto_IVS.md` — a análise já rodada em
  24/08/2026, cujos itens 13, 14 e 15 do checklist são o escopo deste notebook.

**Onde mora a matemática:** `src/ivs_censo/fatorial.py`, testado em
`tests/test_fatorial.py`. Este notebook chama, interpreta e produz figuras — não
reimplementa nada.

**O que este notebook NÃO decide.** Quatro pontos são da orientação, e aqui se produz a
evidência e o custo de cada opção, não a escolha: o destino do indicador de lixo, o
número de fatores, a política do sigilo no analfabetismo, e se a solução oficial será
ortogonal ou oblíqua.

**Saídas:** `banco_de_dados/eda/fatorial/nb04_*.csv` e
`banco_de_dados/eda/fatorial/figuras/*.png`.

## 1. Carga e recorte

O recorte é o mesmo do Notebook 02 — `urbano = 1` e `Dados_sig = 'OK'` — e as fórmulas dos
indicadores vêm de `src/ivs_censo/indicadores.py`, nunca redefinidas aqui.

A **renda entra invertida** (−renda) para que todas as variáveis apontem no mesmo sentido:
valor maior = mais vulnerável. A inversão não muda |r|, autovalores, KMO nem
comunalidades; muda o sinal das cargas, e com ele a leitura.

A exclusão de casos é por lista (*listwise*). O custo está medido e é grande: 16.563
setores, 15,9% do recorte, quase todos pelo sigilo do analfabetismo — e a §6.2.6 do
`GUIA_DO_PROJETO.md` documenta que esse sigilo **não é aleatório**, incide nos setores de
melhor situação. O livro da Enap não trata de dados faltantes em nenhuma das 74 páginas;
a lacuna fica declarada no bloco 10.

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os                          # caminhos e criação de pastas
import sys                         # para registrar src/ no caminho de importação
import sqlite3                     # leitura do banco da entrega
import numpy as np                 # álgebra
import pandas as pd                # DataFrames
import matplotlib.pyplot as plt    # figuras
from pathlib import Path           # caminhos como objetos

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)


def _find_project_root():
    """Detecta a raiz do projeto (independente de onde o Jupyter inicia o kernel)."""
    cwd = Path.cwd().resolve()
    for d in [cwd, *cwd.parents]:
        if (d / 'requirements.txt').is_file() and (d / 'dados').is_dir() and (d / 'docs').is_dir():
            return d
    raise RuntimeError(f'Raiz do projeto não encontrada a partir de: {cwd}')

ROOT = _find_project_root()
CAMINHO_DB  = ROOT / 'banco_de_dados' / 'entrega_orientadora' / 'Base_ELSI_70Municipios_Censo2022.db'
CAMINHO_FAT = ROOT / 'banco_de_dados' / 'eda' / 'fatorial'
CAMINHO_FIG = CAMINHO_FAT / 'figuras'
CAMINHO_FIG.mkdir(parents=True, exist_ok=True)

# A matemática vive no módulo, testada. Aqui só se chama.
sys.path.insert(0, str(ROOT / 'src'))
from ivs_censo.fatorial import (IVS7, ROTULOS, acp, bartlett, bootstrap_cargas,   # noqa: E402
                                comunalidades_obliquas, escores_regressao,
                                fatoracao_eixo_principal, horn, kmo,
                                matriz_correlacao, postos, rotacao_promax, smc, varimax)

# Paleta do projeto — a mesma dos decks e dos relatórios.
TINTA, PETROL, CLAY, CINZA = '#1A1A1A', '#1F4E4A', '#A83A2C', '#666666'
SURF = '#FCFCFB'

# NOTA SOBRE COR: petrol e clay, usados como par categórico, ficam a ΔE 6,1 em
# protanopia — abaixo do piso de 8. A paleta é a identidade do projeto e não muda aqui;
# as figuras é que foram desenhadas para não depender de cor sozinha: rótulo direto em
# cada marca, traço cheio contra tracejado, e valor impresso em cada célula do mapa de
# cargas. Cor é reforço, nunca o único canal.

# Eixos em português: o separador decimal é vírgula, como no resto dos documentos.
VIRGULA = plt.matplotlib.ticker.FuncFormatter(lambda v, _: f'{v:g}'.replace('.', ','))


def salvar(fig, nome):
    """Grava a figura em .../fatorial/figuras/ no padrão do projeto (dpi 150)."""
    caminho = CAMINHO_FIG / nome
    fig.savefig(caminho, dpi=150, bbox_inches='tight', facecolor=SURF)
    plt.close(fig)
    print(f'figura: {caminho.relative_to(ROOT)}')

print(f'Raiz do projeto: {ROOT}')

Raiz do projeto: /Users/pedro/Documents/Iniciacao Cientifica/Projeto_IVS_Censo22


In [2]:
# ── Carga do banco da entrega ────────────────────────────────────────────────
# CD_TIPO entra agora porque o bloco 9 precisa dele (CD_TIPO = 1 marca Favela e
# Comunidade Urbana). Lê-lo aqui evita reabrir o banco lá na frente.
COLUNAS = ['CD_SETOR', 'NM_MUN', 'CD_TIPO', 'urbano', 'Dados_sig'] + IVS7

con = sqlite3.connect(CAMINHO_DB)
try:
    bruto = pd.read_sql(f"SELECT {', '.join(COLUNAS)} FROM setores_censitarios", con)
finally:
    con.close()

df = bruto[(bruto['urbano'].astype(str) == '1') & (bruto['Dados_sig'] == 'OK')].copy()
df['renda_inv'] = -df['renda_media']          # sentido único: maior = mais vulnerável

# Os dois conjuntos de variáveis que o notebook compara o tempo todo.
IVS7_INV = [c if c != 'renda_media' else 'renda_inv' for c in IVS7]   # 7 componentes
IVS6 = [c for c in IVS7_INV if c != 'pct_lixo_inad']                  # sem o lixo
NOMES7 = [ROTULOS[c] for c in IVS7_INV]
NOMES6 = [ROTULOS[c] for c in IVS6]

completo7 = df[IVS7_INV].dropna()
completo6 = df[IVS6].dropna()

print(f'setores no banco            : {len(bruto):,}')
print(f'recorte urbano + Dados_sig OK: {len(df):,}')
print(f'completos nas 7 variáveis    : {len(completo7):,}')
print(f'completos nas 6 (sem lixo)   : {len(completo6):,}')
print(f'perdidos por listwise        : {len(df) - len(completo7):,} '
      f'({100*(len(df)-len(completo7))/len(df):.1f}%)')

# TRAVA: os dois números que todos os documentos do projeto afirmam. Se não baterem,
# a base mudou e nada abaixo é comparável com o que já foi publicado.
assert len(df) == 104_108, f'recorte deu {len(df):,}, esperado 104.108'
assert len(completo7) == 87_545, f'completos deu {len(completo7):,}, esperado 87.545'
print('\nconferência OK — recorte e casos completos batem com a documentação')

setores no banco            : 109,032
recorte urbano + Dados_sig OK: 104,108
completos nas 7 variáveis    : 87,545
completos nas 6 (sem lixo)   : 87,545
perdidos por listwise        : 16,563 (15.9%)

conferência OK — recorte e casos completos batem com a documentação


## 2. Adequabilidade da base

Etapa 1 do livro (p. 39–46). Reproduz o diagnóstico de 24/08/2026 e acrescenta a **SMC**
por variável, que a p. 42 recomenda como diagnóstico complementar.

**Correlação de Spearman como referência**, com Pearson como sensibilidade. A escolha está
fora do catálogo de correlações do livro (p. 11–15), que cobre combinações de variáveis
categóricas — as sete aqui são contínuas, o que pelo catálogo levaria a Pearson. A
justificativa é a não-normalidade documentada na §9 do relatório da EDA: assimetria de
3,42 na água e 3,74 na renda, curtose de 49,5 na renda. O custo da escolha é grande e está
medido na tabela comparativa abaixo — com Pearson a base **reprovaria** em dois critérios.

**TRAVA:** os números têm de reproduzir `resumo_adequabilidade.csv` exatamente. Se
divergirem, há erro de migração e não se deve seguir.

**Duas leituras a registrar:**

1. O **Bartlett é vazio nesta escala**. A p. 43 adverte que o teste "depende muito do
   tamanho amostral e tende a rejeitar a hipótese nula para amostras grandes". Com 87 mil
   setores, rejeitar H₀ não é evidência de estrutura: é evidência de n. A conclusão de
   adequabilidade se apoia no **KMO e nos MSA individuais**, que não crescem com o n.
2. **Multicolinearidade: o número que circula nos documentos é de outra matriz.** O
   relatório da EDA (§9) e, a partir dele, o guia de leitura e o `GUIA_DO_PROJETO.md`
   citam renda × cor/raça a **−0,811** e o comparam ao limiar de 0,80 da p. 42, acima do
   qual "fica inviável separar o peso delas em cada um dos fatores". Esse −0,811 foi
   calculado **par a par**, cada coeficiente sobre os seus próprios casos completos, nos
   104.108 setores do recorte. A matriz que a análise fatorial decompõe é **listwise**,
   nos 87.545 setores completos nas sete variáveis — e nela o mesmo par dá **0,784**.
   Abaixo do limiar. A diferença é inteiramente o conjunto de setores, e vai na mesma
   direção do viés do sigilo: a base listwise perde os setores de melhor situação, o que
   comprime a associação entre renda e cor/raça. Nenhum par da matriz fatorada chega a
   0,80. Isso não dissolve a objeção — 0,784 continua alto e o bloco socioeconômico
   continua coeso — mas o texto do artigo precisa citar o número da matriz que foi de
   fato fatorada, e dizer qual é qual.

In [3]:
# ── Matrizes de correlação e medidas de adequabilidade ───────────────────────
def adequabilidade(X, nomes, metodo='spearman'):
    """Roda a Etapa 1 do livro sobre um conjunto de variáveis já sem faltantes."""
    R = X.corr(method=metodo).to_numpy()
    n, p = X.shape
    fora = R[~np.eye(p, dtype=bool)]
    kmo_global, msa = kmo(R)
    qui, gl, pval = bartlett(R, n)
    return {
        'R': pd.DataFrame(R, index=nomes, columns=nomes),
        'n': n, 'p': p, 'razao': n / p,
        'pct_acima_030': float((np.abs(fora) >= 0.30).mean()),
        'kmo': kmo_global, 'msa': pd.Series(msa, index=nomes),
        'smc': pd.Series(smc(R), index=nomes),
        'qui2': qui, 'gl': gl, 'p_valor': pval,   # 'p' acima é o nº de variáveis
    }

a7s = adequabilidade(completo7, NOMES7, 'spearman')
a7p = adequabilidade(completo7, NOMES7, 'pearson')
a6s = adequabilidade(completo6, NOMES6, 'spearman')

# TRAVA contra resumo_adequabilidade.csv (gerado em 24/08/2026 a partir do CSV da entrega).
assert round(a7s['kmo'], 4) == 0.7826, f"KMO deu {a7s['kmo']:.4f}"
assert round(float(a7s['msa'].min()), 4) == 0.6995, f"MSA mín deu {a7s['msa'].min():.4f}"
assert round(a7s['qui2'], 4) == 235084.3838, f"Bartlett deu {a7s['qui2']:.4f}"
print('TRAVA OK — KMO 0,7826 · MSA mín 0,6995 · Bartlett 235.084,3838 reproduzidos\n')

comparativo = pd.DataFrame({
    '7 comp. Spearman': [a7s['n'], a7s['razao'], 100*a7s['pct_acima_030'], a7s['kmo'],
                         a7s['msa'].min(), a7s['qui2']],
    '7 comp. Pearson':  [a7p['n'], a7p['razao'], 100*a7p['pct_acima_030'], a7p['kmo'],
                         a7p['msa'].min(), a7p['qui2']],
    '6 comp. Spearman': [a6s['n'], a6s['razao'], 100*a6s['pct_acima_030'], a6s['kmo'],
                         a6s['msa'].min(), a6s['qui2']],
}, index=['n', 'casos/variável', '% |r| >= 0,30', 'KMO', 'MSA mínimo', 'Bartlett qui2'])
print(comparativo.round(3).to_string())

TRAVA OK — KMO 0,7826 · MSA mín 0,6995 · Bartlett 235.084,3838 reproduzidos

                7 comp. Spearman  7 comp. Pearson  6 comp. Spearman
n                      87545.000        87545.000         87545.000
casos/variável         12506.429        12506.429         14590.833
% |r| >= 0,30             57.143           33.333            80.000
KMO                        0.783            0.732             0.787
MSA mínimo                 0.700            0.542             0.715
Bartlett qui2         235084.384       131592.709        225320.177


In [4]:
# ── SMC, MSA e o cheque de multicolinearidade ────────────────────────────────
# A SMC (p. 42) mede quanto da variabilidade de cada variável as demais explicam. Perto de
# zero indica variável independente das outras — candidata a sair; perto de um indica
# redundância. É a estimativa inicial de comunalidade que o eixo principal usa no bloco 4.
diag7 = pd.DataFrame({'MSA': a7s['msa'], 'SMC': a7s['smc']})
diag6 = pd.DataFrame({'MSA': a6s['msa'], 'SMC': a6s['smc']})
print('Sete componentes:');  print(diag7.round(3).to_string())
print('\nSeis componentes (sem lixo):'); print(diag6.round(3).to_string())

# Pares acima do limiar de multicolinearidade da p. 42.
R7 = a7s['R']
pares = [(R7.index[i], R7.columns[j], R7.iloc[i, j])
         for i in range(len(R7)) for j in range(i+1, len(R7))
         if abs(R7.iloc[i, j]) >= 0.80]
print('\nPares com |r| >= 0,80 na matriz FATORADA (listwise, n = 87.545) — limiar da p. 42:')
for a, b, r in pares:
    print(f'  {a} × {b}: {r:.3f}')
if not pares:
    print('  nenhum')

# O par que os documentos citam, nas duas matrizes — a divergência é de conjunto de casos,
# não de método, e precisa aparecer com as duas procedências.
par_listwise = R7.loc['Renda (invertida)', 'Cor/raça PPI']
eda = pd.read_csv(ROOT / 'banco_de_dados' / 'eda' / 'correlacao_spearman.csv',
                  sep=';', index_col=0, encoding='utf-8-sig')
par_pairwise = abs(eda.loc['renda_media', 'pct_raca_pretpardind'])
print(f'\nrenda × cor/raça, em módulo:')
print(f'  matriz fatorada (listwise, n = {len(completo7):,}) : {par_listwise:.3f}')
print(f'  EDA §9 (par a par, recorte de {len(df):,})        : {par_pairwise:.3f}')
print(f'  limiar de multicolinearidade (livro, p. 42)     : 0,800')
print('  -> na matriz que a fatorial decompõe, o par NÃO cruza o limiar')

# Grava o bloco de adequabilidade.
saida_adeq = pd.concat([
    diag7.assign(cenario='ivs7_spearman'),
    diag6.assign(cenario='ivs6_sem_lixo_spearman'),
]).rename_axis('variavel').reset_index()
saida_adeq.round(4).to_csv(CAMINHO_FAT / 'nb04_adequabilidade.csv',
                           sep=';', index=False, encoding='utf-8-sig')
a7s['R'].round(3).to_csv(CAMINHO_FAT / 'nb04_correlacao_ivs7_spearman.csv',
                         sep=';', encoding='utf-8-sig')
print('\nnb04_adequabilidade.csv · nb04_correlacao_ivs7_spearman.csv')

Sete componentes:
                      MSA    SMC
Água inadequada     0.753  0.222
Esgoto inadequado   0.834  0.333
Lixo inadequado     0.700  0.106
Razão de moradores  0.857  0.261
Analfabetismo 15+   0.819  0.597
Renda (invertida)   0.724  0.736
Cor/raça PPI        0.782  0.653

Seis componentes (sem lixo):
                      MSA    SMC
Água inadequada     0.745  0.222
Esgoto inadequado   0.850  0.317
Razão de moradores  0.913  0.233
Analfabetismo 15+   0.815  0.597
Renda (invertida)   0.715  0.735
Cor/raça PPI        0.780  0.648

Pares com |r| >= 0,80 na matriz FATORADA (listwise, n = 87.545) — limiar da p. 42:
  nenhum

renda × cor/raça, em módulo:
  matriz fatorada (listwise, n = 87,545) : 0.784
  EDA §9 (par a par, recorte de 104,108)        : 0.811
  limiar de multicolinearidade (livro, p. 42)     : 0,800
  -> na matriz que a fatorial decompõe, o par NÃO cruza o limiar

nb04_adequabilidade.csv · nb04_correlacao_ivs7_spearman.csv


## 3. Número de fatores

Três critérios, e o livro (p. 28–32) manda usá-los em conjunto porque costumam discordar.
Aqui eles discordam — e é preciso dizer isso com todas as letras.

**O critério de Kaiser é o mais fraco neste caso.** A p. 29 registra que ele funciona
melhor entre 20 e 50 variáveis (Tabachnick & Fidell) e que é mais preciso com comunalidades
acima de 0,7 (Stevens). Aqui são 6 ou 7 variáveis, com comunalidade mínima de 0,380. E é
justamente Kaiser quem sustenta o segundo fator na solução de 7 variáveis, decidindo com
folga de 0,09 entre o segundo autovalor (1,049) e o terceiro (0,958).

**Na solução sem o lixo, Kaiser retém UM fator** — o segundo autovalor é 0,9585, logo
abaixo de 1 — e a análise paralela de Horn concorda. A retenção do segundo fator, o de
saneamento, passa a se apoiar na **razão teórica**: as duas dimensões vêm do IVS-BH 2012.
A p. 32 legitima isso ao dizer que a decisão final pode ser teórica e que a pergunta certa
é "teoricamente faz mais sentido essas variáveis estarem agrupadas em quantos fatores?".
É escolha declarada, não número escondido.

A **análise paralela de Horn** (1965) não está no livro da Enap — está em nota de rodapé em
Figueiredo & Silva (2010). É um critério mais moderno e mais robusto que Kaiser, e o
projeto o usa desde 24/08/2026.

In [5]:
# ── Autovalores, Kaiser, Horn e variância acumulada ──────────────────────────
def espectro(adeq, k=2):
    """Autovalores observados contra os de dados aleatórios de mesmo tamanho (Horn)."""
    R = adeq['R'].to_numpy()
    val, _ = acp(R, k)
    # o script original limita a simulação a 20 mil linhas: com p pequeno os autovalores
    # aleatórios já estabilizaram, e o custo cai. Mantido para reproduzir o que existe.
    hval = horn(min(adeq['n'], 20000), adeq['p'])
    return pd.DataFrame({
        'componente': np.arange(1, adeq['p'] + 1),
        'autovalor': val,
        'pct_variancia': 100 * val / adeq['p'],
        'pct_acumulado': 100 * np.cumsum(val) / adeq['p'],
        'horn_aleatorio': hval,
        'passa_kaiser': val > 1,
        'passa_horn': val > hval,
    })

esp7, esp6 = espectro(a7s), espectro(a6s)
print('Sete componentes (Spearman):'); print(esp7.round(4).to_string(index=False))
print('\nSeis componentes, sem o lixo (Spearman):'); print(esp6.round(4).to_string(index=False))

for nome, esp in (('7 componentes', esp7), ('6 sem lixo', esp6)):
    print(f"\n{nome}: Kaiser retém {int(esp['passa_kaiser'].sum())} · "
          f"Horn retém {int(esp['passa_horn'].sum())} · "
          f"variância acumulada com 2 fatores = {esp['pct_acumulado'].iloc[1]:.1f}%")

pd.concat([esp7.assign(cenario='ivs7_spearman'), esp6.assign(cenario='ivs6_sem_lixo_spearman')]) \
  .round(4).to_csv(CAMINHO_FAT / 'nb04_autovalores.csv', sep=';', index=False, encoding='utf-8-sig')

Sete componentes (Spearman):
 componente  autovalor  pct_variancia  pct_acumulado  horn_aleatorio  passa_kaiser  passa_horn
          1     3.2997        47.1383        47.1383          1.0259          True        True
          2     1.0486        14.9794        62.1177          1.0155          True        True
          3     0.9575        13.6788        75.7966          1.0075         False       False
          4     0.6198         8.8541        84.6507          1.0005         False       False
          5     0.5544         7.9205        92.5712          0.9927         False       False
          6     0.3478         4.9683        97.5395          0.9848         False       False
          7     0.1722         2.4605       100.0000          0.9731         False       False

Seis componentes, sem o lixo (Spearman):
 componente  autovalor  pct_variancia  pct_acumulado  horn_aleatorio  passa_kaiser  passa_horn
          1     3.2391        53.9858        53.9858          1.0232      

In [6]:
# ── Figura: scree plot com a linha de Horn ───────────────────────────────────
# FORMA: duas séries num eixo só — autovalor observado contra o que dados sem estrutura
# nenhuma produziriam. É *ênfase*, não categórico: o observado é a linha que importa
# (clay, traço cheio, marcador grande), Horn é referência (cinza, tracejado). A linha de
# Kaiser em 1,0 é recessiva, pontilhada. Cada série leva rótulo direto além da legenda —
# quem não distingue as cores lê o rótulo.
fig, eixos = plt.subplots(1, 2, figsize=(11, 4.2), facecolor=SURF)
for ax, esp, titulo in ((eixos[0], esp7, '7 componentes'),
                        (eixos[1], esp6, '6 componentes (sem o lixo)')):
    x = esp['componente']
    ax.axhline(1.0, color=CINZA, lw=1, ls=':', zorder=1)
    ax.plot(x, esp['horn_aleatorio'], color=CINZA, lw=2, ls='--', marker='o',
            ms=5, mfc=SURF, mec=CINZA, zorder=2)
    ax.plot(x, esp['autovalor'], color=CLAY, lw=2, marker='o', ms=8,
            mfc=CLAY, mec=SURF, mew=1.5, zorder=3)
    # Os rótulos diretos vão onde as duas linhas estão BEM separadas — no primeiro
    # componente e na cauda. No meio elas se cruzam, e rótulo ali vira colisão.
    ax.annotate('autovalor observado', (x.iloc[0], esp['autovalor'].iloc[0]),
                textcoords='offset points', xytext=(14, -3), color=CLAY, fontsize=9)
    ax.annotate('acaso (Horn)', (x.iloc[4], esp['horn_aleatorio'].iloc[4]),
                textcoords='offset points', xytext=(0, 11), color=CINZA, fontsize=9,
                ha='center')
    ax.annotate('Kaiser = 1', (x.iloc[-1], 1.0), textcoords='offset points',
                xytext=(-2, -15), color=CINZA, fontsize=8, ha='right')
    ax.set_title(titulo, color=TINTA, fontsize=11, loc='left')
    ax.set_xlabel('componente', color=CINZA, fontsize=9)
    ax.set_xticks(x)
    ax.tick_params(colors=CINZA, labelsize=9)
    ax.yaxis.set_major_formatter(VIRGULA)
    ax.grid(axis='y', color=CINZA, alpha=0.15, lw=0.8)
    ax.set_axisbelow(True)
    for lado in ('top', 'right'):
        ax.spines[lado].set_visible(False)
    for lado in ('left', 'bottom'):
        ax.spines[lado].set_color(CINZA)
    ax.set_facecolor(SURF)
eixos[0].set_ylabel('autovalor', color=CINZA, fontsize=9)
# Legenda com alças explícitas: passar uma lista de rótulos faria a linha de Kaiser
# entrar como primeira entrada, sem nome, e deslocaria os outros dois.
from matplotlib.lines import Line2D
eixos[0].legend(handles=[
    Line2D([], [], color=CLAY, lw=2, marker='o', ms=7, mfc=CLAY, mec=SURF, mew=1.2,
           label='autovalor observado'),
    Line2D([], [], color=CINZA, lw=2, ls='--', marker='o', ms=5, mfc=SURF, mec=CINZA,
           label='acaso (Horn)'),
], frameon=False, fontsize=8.5, labelcolor=TINTA,
    # longe do rótulo direto do primeiro componente, que fica no alto à esquerda
    loc='center right', bbox_to_anchor=(1.0, 0.63))
fig.suptitle('Quantos fatores reter — observado contra o acaso',
             color=TINTA, fontsize=12, x=0.007, ha='left', y=1.02)
salvar(fig, 'nb04_scree_horn.png')

figura: banco_de_dados/eda/fatorial/figuras/nb04_scree_horn.png


## 4. Extração comparada — ACP e eixo principal

A p. 27 traz duas regras sobre quando ACP e análise fatorial dão a mesma coisa: Hair
(acima de 30 variáveis, **ou** comunalidades acima de 0,60 na maioria) e Stevens (com 30 ou
mais variáveis e comunalidades acima de 0,7 em todas, as soluções ficam muito próximas;
**abaixo de 20 variáveis e com comunalidades baixas, abaixo de 0,4, podem divergir**).

O projeto tem 6 variáveis e comunalidade mínima de 0,380. Cai no lado da divergência. Isso
significa que "ACP e AF dariam o mesmo" deixou de ser pressuposto e virou **hipótese a
testar** — e o teste é barato.

A diferença entre as duas está inteiramente na diagonal da matriz decomposta: a ACP põe 1,
usando toda a variância de cada variável; a análise fatorial põe a comunalidade, usando só
a variância compartilhada. Como a comunalidade só se conhece depois de extrair, o eixo
principal itera a partir da SMC.

In [7]:
# ── ACP e fatoração do eixo principal, lado a lado ───────────────────────────
def comparar_extracao(adeq, nomes, k=2):
    R = adeq['R'].to_numpy()
    _, c_acp = acp(R, k)
    c_paf, h_paf, info = fatoracao_eixo_principal(R, k)
    # o sinal do autovetor é arbitrário: alinha o PAF ao ACP coluna a coluna antes de
    # comparar, senão a diferença mediria troca de sinal, não divergência de método.
    for j in range(k):
        if c_paf[:, j] @ c_acp[:, j] < 0:
            c_paf[:, j] = -c_paf[:, j]
    tab = pd.DataFrame({
        **{f'ACP_{j+1}': c_acp[:, j] for j in range(k)},
        **{f'PAF_{j+1}': c_paf[:, j] for j in range(k)},
        **{f'dif_{j+1}': np.abs(c_acp[:, j] - c_paf[:, j]) for j in range(k)},
        'comun_ACP': (c_acp ** 2).sum(axis=1),
        'comun_PAF': h_paf,
        'SMC': adeq['smc'].to_numpy(),
    }, index=nomes)
    return tab, info

ext6, info6 = comparar_extracao(a6s, NOMES6)
ext7, info7 = comparar_extracao(a7s, NOMES7)

for nome, tab, info in (('6 componentes (sem lixo)', ext6, info6),
                        ('7 componentes', ext7, info7)):
    difs = tab[[c for c in tab.columns if c.startswith('dif_')]]
    pior = difs.stack().idxmax()
    print(f'\n── {nome} ──')
    print(tab.round(3).to_string())
    print(f"convergiu em {info['iteracoes']} iterações (delta {info['delta']:.2e}) · "
          f"caso de Heywood: {info['heywood']}")
    print(f'maior diferença absoluta: {difs.to_numpy().max():.3f} '
          f'em {pior[0]} / {pior[1]}')

pd.concat([ext6.assign(cenario='ivs6_sem_lixo_spearman'),
           ext7.assign(cenario='ivs7_spearman')]).rename_axis('variavel') \
  .round(4).to_csv(CAMINHO_FAT / 'nb04_extracao_comparada.csv', sep=';', encoding='utf-8-sig')
print('\nnb04_extracao_comparada.csv')


── 6 componentes (sem lixo) ──
                    ACP_1  ACP_2  PAF_1  PAF_2  dif_1  dif_2  comun_ACP  comun_PAF    SMC
Água inadequada    -0.503  0.754 -0.432  0.513  0.071  0.241      0.822      0.450  0.222
Esgoto inadequado  -0.666  0.413 -0.576  0.315  0.090  0.098      0.615      0.431  0.317
Razão de moradores -0.616  0.023 -0.503  0.086  0.113  0.062      0.380      0.260  0.233
Analfabetismo 15+  -0.825 -0.299 -0.783 -0.159  0.042  0.140      0.770      0.638  0.597
Renda (invertida)  -0.871 -0.312 -0.912 -0.293  0.041  0.019      0.856      0.918  0.735
Cor/raça PPI       -0.850 -0.178 -0.814 -0.067  0.036  0.111      0.754      0.666  0.648
convergiu em 192 iterações (delta 9.93e-08) · caso de Heywood: False
maior diferença absoluta: 0.241 em Água inadequada / dif_2

── 7 componentes ──
                    ACP_1  ACP_2  PAF_1  PAF_2  dif_1  dif_2  comun_ACP  comun_PAF    SMC
Água inadequada    -0.501  0.049 -0.421  0.445  0.080  0.396      0.253      0.376  0.222
Esgoto in

## 5. Rotação comparada — Varimax e promax

**A passagem mais consequente do livro para este projeto está na p. 38:** "usar rotação
ortogonal com dados de Ciências Humanas e Sociais não parece ter nenhum sentido. Nessas
áreas, as variáveis quase sempre são correlacionadas. [...] para usar rotação ortogonal, o
pesquisador precisaria ter evidências teóricas ou empíricas muito fortes de que os fatores
não são correlacionados."

A Varimax usada até aqui foi herdada de Figueiredo & Silva (2010), que a adota "por ser a
mais comum". O livro inverte o ônus da prova: ortogonal é o caminho que exige
justificativa, e a justificativa pedida é de que os fatores **não** se correlacionam. No
IVS isso é implausível à partida — territórios pobres têm pior saneamento, e as próprias
correlações mostram: esgoto com analfabetismo a 0,429, esgoto com renda a −0,454.

A rotação oblíqua produz três coisas que a ortogonal não pode produzir:

1. **Φ, a matriz de correlação entre os fatores** — evidência de validação que o projeto
   hoje não tem. Se os dois fatores se correlacionarem positivamente e com magnitude
   moderada, é o que a teoria da vulnerabilidade prevê.
2. **Duas matrizes de cargas** (p. 21–22): a **padrão**, de coeficientes de regressão, e a
   **estrutura**, de correlações. O livro registra que a maioria interpreta a padrão, e é
   dela que saem os pesos. Na solução ortogonal as duas coincidem.
3. **Uma repartição de pesos diferente.** O 65/35 foi calculado por soma dos quadrados de
   cargas ortogonais. Com fatores correlacionados parte da variância é compartilhada, e a
   repartição muda — o número precisa ser recalculado antes de virar peso oficial.

**Cheque obrigatório (p. 22):** numa solução oblíqua uma carga pode passar de 1 sem ser
erro, por ser coeficiente de regressão. Se isso ocorrer, testa-se a variância residual da
variável: negativa, a solução é inadmissível e sugere fatores demais extraídos.

In [8]:
# ── Varimax × promax ─────────────────────────────────────────────────────────
def comparar_rotacao(adeq, nomes, k=2):
    R = adeq['R'].to_numpy()
    _, cargas = acp(R, k)
    vmax = varimax(cargas)
    padrao, estrutura, phi = rotacao_promax(cargas)

    # Convenção de leitura: o sinal do autovetor é arbitrário. Vira-se cada fator para que
    # a soma das cargas fique positiva — assim "carga positiva = mais vulnerável", que é o
    # sentido em que as variáveis já foram postas (renda invertida).
    for M in (vmax,):
        for j in range(k):
            if M[:, j].sum() < 0:
                M[:, j] *= -1
    sinais = np.where(padrao.sum(axis=0) < 0, -1.0, 1.0)
    padrao = padrao * sinais
    estrutura = estrutura * sinais
    phi = phi * np.outer(sinais, sinais)

    comun_ort = (vmax ** 2).sum(axis=1)
    comun_obl = comunalidades_obliquas(padrao, phi)
    tab = pd.DataFrame({
        **{f'Varimax{j+1}': vmax[:, j] for j in range(k)},
        **{f'Padrao{j+1}': padrao[:, j] for j in range(k)},
        **{f'Estrutura{j+1}': estrutura[:, j] for j in range(k)},
        'comun_ortogonal': comun_ort,
        'comun_obliqua': comun_obl,
        'var_residual': 1 - comun_obl,
    }, index=nomes)

    def repartir(M):
        ss = (M ** 2).sum(axis=0)
        return ss / ss.sum()

    return tab, phi, repartir(vmax), repartir(padrao)

rot6, phi6, rep6_ort, rep6_obl = comparar_rotacao(a6s, NOMES6)
rot7, phi7, rep7_ort, rep7_obl = comparar_rotacao(a7s, NOMES7)

print('── 6 componentes, sem o lixo ──')
print(rot6.round(3).to_string())
print(f'\nΦ (correlação entre os fatores):\n{pd.DataFrame(phi6).round(3).to_string()}')
print(f'\nrepartição do peso  ortogonal: {100*rep6_ort[0]:.1f} / {100*rep6_ort[1]:.1f}')
print(f'repartição do peso   oblíqua: {100*rep6_obl[0]:.1f} / {100*rep6_obl[1]:.1f}')
print(f'referência IVS-BH 2012      : 60,0 / 40,0')

# Cheque da p. 22: carga acima de 1 exige variância residual positiva.
altas = rot6[[c for c in rot6.columns if c.startswith('Padrao')]].abs().to_numpy().max()
print(f'\nmaior carga padrão em módulo: {altas:.3f}')
if altas > 1:
    print('  carga acima de 1 — conferindo a variância residual (livro p. 22):')
    print(rot6[['var_residual']].round(4).to_string())
    assert (rot6['var_residual'] > 0).all(), 'variância residual negativa: solução inadmissível'
    print('  todas positivas — solução admissível')
else:
    print('  nenhuma carga acima de 1; o cheque da p. 22 não se aplica')
assert (rot6['var_residual'] > 0).all() and (rot7['var_residual'] > 0).all()

print('\n── 7 componentes ──')
print(rot7.round(3).to_string())
print(f'\nΦ:\n{pd.DataFrame(phi7).round(3).to_string()}')
print(f'repartição ortogonal: {100*rep7_ort[0]:.1f} / {100*rep7_ort[1]:.1f} · '
      f'oblíqua: {100*rep7_obl[0]:.1f} / {100*rep7_obl[1]:.1f}')

rot6.rename_axis('variavel').round(4).to_csv(
    CAMINHO_FAT / 'nb04_cargas_ivs6_sem_lixo.csv', sep=';', encoding='utf-8-sig')
rot7.rename_axis('variavel').round(4).to_csv(
    CAMINHO_FAT / 'nb04_cargas_ivs7.csv', sep=';', encoding='utf-8-sig')
pd.DataFrame(phi6, index=['Fator 1', 'Fator 2'], columns=['Fator 1', 'Fator 2']) \
  .round(4).to_csv(CAMINHO_FAT / 'nb04_phi_ivs6_sem_lixo.csv', sep=';', encoding='utf-8-sig')
print('\nnb04_cargas_ivs6_sem_lixo.csv · nb04_cargas_ivs7.csv · nb04_phi_ivs6_sem_lixo.csv')

── 6 componentes, sem o lixo ──
                    Varimax1  Varimax2  Padrao1  Padrao2  Estrutura1  Estrutura2  comun_ortogonal  comun_obliqua  var_residual
Água inadequada        0.087     0.903   -0.211    0.999       0.310       0.889            0.822          0.822         0.178
Esgoto inadequado      0.392     0.679    0.207    0.656       0.549       0.764            0.615          0.615         0.385
Razão de moradores     0.532     0.312    0.490    0.198       0.593       0.453            0.380          0.380         0.620
Analfabetismo 15+      0.868     0.127    0.930   -0.111       0.872       0.374            0.770          0.770         0.230
Renda (invertida)      0.915     0.137    0.979   -0.113       0.920       0.398            0.856          0.856         0.144
Cor/raça PPI           0.833     0.245    0.850    0.034       0.868       0.477            0.754          0.754         0.246

Φ (correlação entre os fatores):
       0      1
0  1.000  0.522
1  0.522  1.0

In [9]:
# ── Figura: mapa de cargas ───────────────────────────────────────────────────
# FORMA: as cargas são grandezas COM SINAL, entre -1 e 1 — logo, paleta DIVERGENTE:
# dois polos opostos (clay quente = positivo, petrol frio = negativo) com cinza neutro no
# meio. Nunca um arco-íris, nunca uma cor no ponto médio. E cada célula leva o valor
# impresso: é a codificação secundária que a validação de cor exige, e quem lê em preto e
# branco continua lendo a tabela.
def rampa_divergente(v):
    """Interpola cinza-claro -> clay (positivo) ou cinza-claro -> petrol (negativo)."""
    neutro = np.array([0.94, 0.94, 0.93])
    alvo = np.array([0.659, 0.227, 0.173]) if v >= 0 else np.array([0.122, 0.306, 0.290])
    t = min(abs(v), 1.0)
    return tuple(neutro + t * (alvo - neutro))

fig, eixos = plt.subplots(1, 2, figsize=(9.0, 4.2), facecolor=SURF,
                          gridspec_kw={'width_ratios': [1, 1]})
for ax, M, titulo in (
        (eixos[0], rot6[['Varimax1', 'Varimax2']], 'Varimax (ortogonal)'),
        (eixos[1], rot6[['Padrao1', 'Padrao2']], 'promax — matriz padrão (oblíqua)')):
    dados = M.to_numpy()
    for i in range(dados.shape[0]):
        for j in range(dados.shape[1]):
            v = dados[i, j]
            ax.add_patch(plt.Rectangle((j + 0.02, i + 0.02), 0.96, 0.96,
                                       facecolor=rampa_divergente(v), edgecolor=SURF, lw=2))
            ax.text(j + 0.5, i + 0.5, f'{v:.2f}'.replace('-', '−').replace('.', ','),
                    ha='center', va='center', fontsize=9.5,
                    color=SURF if abs(v) > 0.55 else TINTA)
    ax.set_xlim(0, dados.shape[1]); ax.set_ylim(dados.shape[0], 0)
    ax.set_xticks(np.arange(dados.shape[1]) + 0.5)
    ax.set_xticklabels(['Fator 1', 'Fator 2'], fontsize=9, color=TINTA)
    ax.set_yticks(np.arange(dados.shape[0]) + 0.5)
    ax.set_yticklabels(M.index if ax is eixos[0] else [], fontsize=9, color=TINTA)
    ax.tick_params(length=0)
    for lado in ('top', 'right', 'left', 'bottom'):
        ax.spines[lado].set_visible(False)
    ax.set_title(titulo, color=TINTA, fontsize=11, loc='left')
    ax.set_facecolor(SURF)
fig.suptitle('Cargas fatoriais — seis componentes, sem o indicador de lixo',
             color=TINTA, fontsize=12, x=0.007, ha='left', y=1.03)
phi_txt = f'{phi6[0, 1]:.3f}'.replace('.', ',')
fig.text(0.007, -0.05,
         'Tom quente = carga positiva · tom frio = carga negativa · cinza = perto de zero.\n'
         'O valor vai impresso em cada célula: a leitura não depende de distinguir as cores.  '
         f'Correlação entre os fatores (Φ) = {phi_txt}',
         color=CINZA, fontsize=8.5, ha='left', linespacing=1.6)
salvar(fig, 'nb04_mapa_cargas.png')

figura: banco_de_dados/eda/fatorial/figuras/nb04_mapa_cargas.png


## 6. Estabilidade das cargas — bootstrap

A p. 23 acusa os métodos não refinados de serem "muito instável[eis] por depender
fortemente da amostra em particular que está sendo analisada". É uma crítica séria ao
plano do projeto, que prevê o IVS como média ponderada. A resposta honesta não é discordar:
é medir.

Mil reamostragens com reposição, refazendo a matriz de correlação, a extração e a rotação
a cada uma. Cada solução é alinhada à da amostra completa antes de entrar na conta — sem
isso, a troca de sinal ou de ordem dos fatores que o LAPACK faz a cada rodada produziria
cancelamento, e o resultado pareceria instabilidade sendo artefato.

Nem a Enap nem Figueiredo trazem bootstrap. É extensão do projeto.

In [10]:
# ── Bootstrap das cargas e da repartição ─────────────────────────────────────
X6 = completo6.to_numpy()
bs = bootstrap_cargas(X6, k=2, n_rep=1000, seed=42, metodo='spearman', rotacao='varimax')

lo_c, hi_c = bs['cargas_ic']
ic_cargas = pd.DataFrame({
    'carga_F1': bs['cargas'][:, 0], 'IC95_F1_inf': lo_c[:, 0], 'IC95_F1_sup': hi_c[:, 0],
    'largura_F1': hi_c[:, 0] - lo_c[:, 0],
    'carga_F2': bs['cargas'][:, 1], 'IC95_F2_inf': lo_c[:, 1], 'IC95_F2_sup': hi_c[:, 1],
    'largura_F2': hi_c[:, 1] - lo_c[:, 1],
}, index=NOMES6)
print(ic_cargas.round(4).to_string())

lo_p, hi_p = bs['repartição_ic']
print(f"\nrepartição do peso entre as dimensões (1.000 reamostragens, semente 42):")
print(f"  Fator 1: {100*bs['repartição'][0]:.1f}%  IC95 [{100*lo_p[0]:.1f}; {100*hi_p[0]:.1f}]")
print(f"  Fator 2: {100*bs['repartição'][1]:.1f}%  IC95 [{100*lo_p[1]:.1f}; {100*hi_p[1]:.1f}]")
print(f"  amplitude do IC: {100*(hi_p[0]-lo_p[0]):.2f} pontos percentuais")

ic_cargas.rename_axis('variavel').round(4).to_csv(
    CAMINHO_FAT / 'nb04_bootstrap_cargas.csv', sep=';', encoding='utf-8-sig')
print('\nnb04_bootstrap_cargas.csv')

                    carga_F1  IC95_F1_inf  IC95_F1_sup  largura_F1  carga_F2  IC95_F2_inf  IC95_F2_sup  largura_F2
Água inadequada      -0.0869      -0.0908      -0.0824      0.0083    0.9025       0.8991       0.9055      0.0063
Esgoto inadequado    -0.3921      -0.3997      -0.3835      0.0162    0.6792       0.6706       0.6881      0.0175
Razão de moradores   -0.5322      -0.5445      -0.5187      0.0258    0.3118       0.2932       0.3337      0.0405
Analfabetismo 15+    -0.8684      -0.8705      -0.8660      0.0046    0.1266       0.1213       0.1333      0.0120
Renda (invertida)    -0.9152      -0.9170      -0.9132      0.0038    0.1369       0.1324       0.1422      0.0099
Cor/raça PPI         -0.8330      -0.8354      -0.8303      0.0051    0.2446       0.2391       0.2512      0.0121

repartição do peso entre as dimensões (1.000 reamostragens, semente 42):
  Fator 1: 65.0%  IC95 [64.7; 65.3]
  Fator 2: 35.0%  IC95 [34.7; 35.3]
  amplitude do IC: 0.59 pontos percentuais

nb04_

## 7. Pesos e escores

Dois caminhos para transformar cargas em índice, e o livro é explícito sobre a diferença
(p. 22–26):

- **Índice 0–1 por média ponderada** das variáveis padronizadas. É o plano do projeto e é
  o que o IVS-BH 2012 faz. Na taxonomia da p. 23 é um método **não refinado**, com três
  defeitos declarados: depende da escala das variáveis, depende da rotação, e é instável
  entre amostras.
- **Escore pelo método da regressão**, B = R⁻¹A (p. 25). Refinado, estável, mas sai numa
  escala padronizada abstrata, perde a comparabilidade com o IVS-BH e exige a matriz R⁻¹
  para ser reproduzido por terceiros.

A saída não é escolher, é **calcular os dois e usar a concordância como validação**. Se os
dois ordenarem os setores do mesmo jeito, a instabilidade que o livro teme não se
materializou nesta amostra, e o índice 0–1 fica reportado com a evidência no apêndice.

**Composição dos pesos.** Cada variável é atribuída à dimensão em que carrega mais alto; o
peso da dimensão é a soma dos quadrados das cargas daquele fator sobre o total; dentro da
dimensão, o peso de cada variável é proporcional ao quadrado da sua carga. É a formulação
que a p. 71 defende: "os itens contribuem de maneira desigual para o fator: quanto maior a
carga fatorial, maior a contribuição do item", ao contrário das técnicas mais simples que
pressupõem contribuição igual.

**A padronização aqui é min-max global, e é provisória.** A normalização definitiva é
**por município** e pertence ao Notebook 03. Está medido que normalizar antes de fatorar
derruba o KMO de 0,783 para 0,720 e muda a repartição de 65/35 para 56/44 — por isso a
fatorial roda sobre os brutos. O índice calculado aqui serve para comparar cenários, não
é o IVS final.

In [11]:
# ── Dos carregamentos aos pesos ──────────────────────────────────────────────
def montar_pesos(cargas_df, colunas, nomes, rep):
    """Atribui cada variável à dimensão de maior carga e reparte o peso dentro dela."""
    M = cargas_df.to_numpy()
    dimensao = np.abs(M).argmax(axis=1)                 # a qual fator cada variável pertence
    peso = np.zeros(len(nomes))
    for j in range(M.shape[1]):
        membros = dimensao == j
        if not membros.any():
            continue
        dentro = M[membros, j] ** 2
        peso[membros] = rep[j] * dentro / dentro.sum()   # peso da dimensão × parte na dimensão
    return pd.DataFrame({'coluna': colunas, 'dimensao': dimensao + 1,
                         'carga': M[np.arange(len(nomes)), dimensao],
                         'peso': peso}, index=nomes)

pesos = montar_pesos(rot6[['Varimax1', 'Varimax2']], IVS6, NOMES6, rep6_ort)
print('Pesos — solução Varimax, 6 componentes, pesos empíricos:')
print(pesos.round(4).to_string())
print(f"\nsoma dos pesos = {pesos['peso'].sum():.6f}")
print(f"dimensão 1 = {100*pesos.loc[pesos['dimensao']==1,'peso'].sum():.1f}%  ·  "
      f"dimensão 2 = {100*pesos.loc[pesos['dimensao']==2,'peso'].sum():.1f}%")
pesos.rename_axis('variavel').round(6).to_csv(
    CAMINHO_FAT / 'nb04_pesos.csv', sep=';', encoding='utf-8-sig')

Pesos — solução Varimax, 6 componentes, pesos empíricos:
                                  coluna  dimensao   carga    peso
Água inadequada            pct_agua_inad         2  0.9025  0.2232
Esgoto inadequado        pct_esgoto_inad         2  0.6792  0.1264
Razão de moradores       razao_moradores         1  0.5322  0.0717
Analfabetismo 15+            pct_analfab         1  0.8684  0.1909
Renda (invertida)              renda_inv         1  0.9152  0.2121
Cor/raça PPI        pct_raca_pretpardind         1  0.8330  0.1757

soma dos pesos = 1.000000
dimensão 1 = 65.0%  ·  dimensão 2 = 35.0%


In [12]:
# ── Índice 0-1 e escore refinado, e a concordância entre os dois ─────────────
def minmax(df_, colunas):
    """Min-max global, provisório: a normalização por município é do Notebook 03."""
    sub = df_[colunas]
    return (sub - sub.min()) / (sub.max() - sub.min())

base6 = df.dropna(subset=IVS6).copy()                 # os 87.545 completos, com CD_SETOR
Z01 = minmax(base6, IVS6)
base6['indice_01'] = (Z01 * pesos['peso'].to_numpy()).sum(axis=1)

# Escore refinado: B = R^-1 A (livro, p. 25).
#
# ATENÇÃO À ESCALA. O modelo foi estimado sobre a matriz de SPEARMAN — ou seja, sobre os
# POSTOS das variáveis, não sobre elas. Os coeficientes B, portanto, se aplicam aos postos
# padronizados. Aplicá-los aos valores brutos padronizados é erro de categoria, e o
# sintoma é imediato e verificável: a variância dos escores deixa de ser 1. As duas
# versões estão calculadas abaixo, e a diferença entre elas é a medida do que a limitação
# 2 do bloco 10 custa.
R6 = a6s['R'].to_numpy()
B = escores_regressao(R6, rot6[['Varimax1', 'Varimax2']].to_numpy())

postos6 = pd.DataFrame(postos(base6[IVS6].to_numpy()), index=base6.index, columns=IVS6)
Zr = (postos6 - postos6.mean()) / postos6.std(ddof=0)        # postos padronizados (coerente)
Zb = (base6[IVS6] - base6[IVS6].mean()) / base6[IVS6].std(ddof=0)   # brutos (incoerente)

escores = Zr.to_numpy() @ B
escores_brutos = Zb.to_numpy() @ B
base6['escore_F1'], base6['escore_F2'] = escores[:, 0], escores[:, 1]
# O escore composto usa a mesma repartição entre dimensões que o índice 0-1.
base6['escore_composto'] = escores @ rep6_ort
base6['escore_composto_bruto'] = escores_brutos @ rep6_ort

print(f'variância dos escores sobre POSTOS  : F1 = {escores[:,0].var():.4f} · '
      f'F2 = {escores[:,1].var():.4f}   <- tem de dar 1')
print(f'variância dos escores sobre BRUTOS  : F1 = {escores_brutos[:,0].var():.4f} · '
      f'F2 = {escores_brutos[:,1].var():.4f}')
assert np.allclose([escores[:,0].var(), escores[:,1].var()], 1.0, atol=1e-6), \
    'escores sobre postos deveriam ter variância 1 — B ou a padronização estão errados'
print(f"índice 0-1: min {base6['indice_01'].min():.4f} · mediana "
      f"{base6['indice_01'].median():.4f} · máx {base6['indice_01'].max():.4f}")

rho = base6[['indice_01', 'escore_composto']].corr(method='spearman').iloc[0, 1]
rho_bruto = base6[['indice_01', 'escore_composto_bruto']].corr(method='spearman').iloc[0, 1]
print(f'\nSpearman índice 0-1 × escore refinado (postos): {rho:.4f}')
print(f'Spearman índice 0-1 × escore refinado (brutos): {rho_bruto:.4f}')
print('critério do plano: acima de 0,95 valida o índice 0-1 apesar de ser "não refinado"')
print('VALIDADO' if rho > 0.95 else 'ABAIXO DO CRITÉRIO — ver a leitura no bloco 10')

base6[['CD_SETOR', 'NM_MUN', 'indice_01', 'escore_F1', 'escore_F2',
       'escore_composto', 'escore_composto_bruto']] \
  .round(6).to_csv(CAMINHO_FAT / 'nb04_escores.csv', sep=';', index=False, encoding='utf-8-sig')
print('nb04_escores.csv')

variância dos escores sobre POSTOS  : F1 = 1.0000 · F2 = 1.0000   <- tem de dar 1
variância dos escores sobre BRUTOS  : F1 = 0.8942 · F2 = 0.9697
índice 0-1: min 0.0647 · mediana 0.3542 · máx 0.8302

Spearman índice 0-1 × escore refinado (postos): 0.9241
Spearman índice 0-1 × escore refinado (brutos): 0.9449
critério do plano: acima de 0,95 valida o índice 0-1 apesar de ser "não refinado"
ABAIXO DO CRITÉRIO — ver a leitura no bloco 10


nb04_escores.csv


## 8. Cenários de decisão

Três decisões estão em aberto e são da orientação. O papel deste bloco é **medir o custo de
cada opção**, não escolher.

A métrica não é variância explicada — é **quantos setores mudam de faixa**. Um índice serve
para classificar território; o que importa é se a escolha metodológica muda a
classificação de quem vai receber política pública.

| Cenário | A decisão que ele mede |
|---|---|
| Dois fatores × um fator | Kaiser e Horn retêm um; a teoria do IVS-BH pede dois |
| Pesos empíricos (65/35) × literatura (60/40) | decisão nº 1 da §6.3 do Guia |
| Com × sem o analfabetismo | a política do sigilo: 16.563 setores em jogo |

In [13]:
# ── Os cenários, comparados por mudança de faixa ─────────────────────────────
def faixas(serie):
    """Quatro faixas por quartil — a categorização prevista para o IVS."""
    return pd.qcut(serie.rank(method='first'), 4, labels=['1 (menor)', '2', '3', '4 (maior)'])

def indice_com_pesos(base_, colunas, vetor_pesos):
    Z = minmax(base_, colunas)
    return (Z * np.asarray(vetor_pesos)).sum(axis=1)

cen = pd.DataFrame(index=base6.index)
cen['dois_fatores'] = base6['indice_01']

# (a) um fator: os pesos passam a ser o quadrado da carga no primeiro componente.
_, c6 = acp(R6, 1)
p1 = (c6[:, 0] ** 2) / (c6[:, 0] ** 2).sum()
cen['um_fator'] = indice_com_pesos(base6, IVS6, p1)

# (b) 60/40 da literatura, mantida a repartição interna de cada dimensão.
pesos_6040 = pesos['peso'].to_numpy().copy()
for j, alvo in ((1, 0.60), (2, 0.40)):
    m = (pesos['dimensao'] == j).to_numpy()
    pesos_6040[m] = alvo * pesos_6040[m] / pesos_6040[m].sum()
cen['pesos_6040'] = indice_com_pesos(base6, IVS6, pesos_6040)

# (c) sem o analfabetismo: refaz a fatorial nas 5 variáveis restantes e recompõe os pesos.
IVS5 = [c for c in IVS6 if c != 'pct_analfab']
base5 = df.dropna(subset=IVS5).copy()
a5 = adequabilidade(base5[IVS5], [ROTULOS[c] for c in IVS5], 'spearman')
_, c5 = acp(a5['R'].to_numpy(), 2)
v5 = varimax(c5)
for j in range(2):
    if v5[:, j].sum() < 0:
        v5[:, j] *= -1
ss5 = (v5 ** 2).sum(axis=0); rep5 = ss5 / ss5.sum()
pesos5 = montar_pesos(pd.DataFrame(v5, index=[ROTULOS[c] for c in IVS5]),
                      IVS5, [ROTULOS[c] for c in IVS5], rep5)
base5['indice_01'] = indice_com_pesos(base5, IVS5, pesos5['peso'].to_numpy())
print(f'cenário sem analfabetismo: {len(base5):,} setores '
      f'(+{len(base5)-len(base6):,} em relação aos {len(base6):,} completos)')
cen['sem_analfab'] = base5.set_index(base5.index)['indice_01'].reindex(base6.index)

# Tabelas de contingência entre as faixas, sempre contra o cenário de referência.
ref = faixas(cen['dois_fatores'])
linhas = []
for nome in ['um_fator', 'pesos_6040', 'sem_analfab']:
    alt = faixas(cen[nome])
    tab = pd.crosstab(ref, alt)
    mudou = int((ref.astype(str) != alt.astype(str)).sum())
    rho_c = cen[['dois_fatores', nome]].corr(method='spearman').iloc[0, 1]
    linhas.append({'cenario': nome, 'setores_que_mudam_de_faixa': mudou,
                   'pct': 100 * mudou / len(cen), 'spearman_com_referencia': rho_c})
    print(f'\n── {nome} × referência (dois fatores, pesos empíricos) ──')
    print(tab.to_string())
    print(f'mudam de faixa: {mudou:,} setores ({100*mudou/len(cen):.1f}%) · '
          f'Spearman {rho_c:.4f}')
    tab.to_csv(CAMINHO_FAT / f'nb04_contingencia_{nome}.csv', sep=';', encoding='utf-8-sig')

resumo_cen = pd.DataFrame(linhas)
resumo_cen.round(4).to_csv(CAMINHO_FAT / 'nb04_cenarios.csv', sep=';', index=False, encoding='utf-8-sig')
print('\n' + resumo_cen.round(4).to_string(index=False))

cenário sem analfabetismo: 104,093 setores (+16,548 em relação aos 87,545 completos)



── um_fator × referência (dois fatores, pesos empíricos) ──
um_fator      1 (menor)      2      3  4 (maior)
dois_fatores                                    
1 (menor)         21479    408      0          0
2                   336  20325   1225          0
3                    49    898  18563       2376
4 (maior)            23    255   2098      19510
mudam de faixa: 7,668 setores (8.8%) · Spearman 0.9830

── pesos_6040 × referência (dois fatores, pesos empíricos) ──
pesos_6040    1 (menor)      2      3  4 (maior)
dois_fatores                                    
1 (menor)         21783    104      0          0
2                   104  21420    362          0
3                     0    362  20892        632
4 (maior)             0      0    632      21254
mudam de faixa: 2,196 setores (2.5%) · Spearman 0.9993

── sem_analfab × referência (dois fatores, pesos empíricos) ──
sem_analfab   1 (menor)      2      3  4 (maior)
dois_fatores                                    
1 (menor)       

## 9. Validação externa — os setores de favela

Esta é a validação mais forte disponível ao projeto, e ela não existe em nenhuma das duas
referências metodológicas: o marcador é **externo ao índice**.

`CD_TIPO = 1` marca os setores de Favela e Comunidade Urbana do Censo 2022, uma
classificação oficial do IBGE, verificada setor a setor contra a lista oficial com 100% de
concordância (§6.2.8 do Guia). Nenhuma das seis variáveis do índice foi usada para
construí-la.

Um IVS bem construído tem de separar esses setores dos demais. Isso é teste de validade de
critério, e vale mais do que qualquer estatística interna à fatorial — KMO, comunalidade e
variância explicada dizem se as variáveis se organizam; só isto diz se o índice acerta.

A área sob a curva ROC é calculada pela estatística de postos de Mann–Whitney, que trata
empates corretamente e dispensa dependência nova.

In [14]:
# ── Separação dos setores de FCU ─────────────────────────────────────────────
base6['fcu'] = pd.to_numeric(base6['CD_TIPO'], errors='coerce').eq(1)
n_fcu, n_out = int(base6['fcu'].sum()), int((~base6['fcu']).sum())
print(f'setores de FCU no conjunto completo: {n_fcu:,} · demais: {n_out:,}')

def auc_postos(escore, positivo):
    """AUC pela estatística de Mann–Whitney: (soma dos postos dos positivos - n1(n1+1)/2) / (n1*n0)."""
    r = pd.Series(escore).rank()                    # postos com média nos empates
    n1 = int(positivo.sum()); n0 = len(escore) - n1
    return float((r[positivo.to_numpy()].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

def curva_roc(escore, positivo, pontos=400):
    """TPR e FPR ao longo de cortes igualmente espaçados nos quantis do escore."""
    cortes = np.quantile(escore, np.linspace(0, 1, pontos))[::-1]
    pos, neg = positivo.to_numpy(), ~positivo.to_numpy()
    tpr = np.array([(escore[pos] >= c).mean() for c in cortes])
    fpr = np.array([(escore[neg] >= c).mean() for c in cortes])
    return np.r_[0, fpr, 1], np.r_[0, tpr, 1]

val = []
for nome, col in (('índice 0-1', 'indice_01'), ('escore refinado', 'escore_composto')):
    s = base6[col].to_numpy()
    a = auc_postos(s, base6['fcu'])
    md_fcu = float(base6.loc[base6['fcu'], col].median())
    md_out = float(base6.loc[~base6['fcu'], col].median())
    val.append({'medida': nome, 'auc': a, 'mediana_fcu': md_fcu,
                'mediana_demais': md_out, 'diferenca': md_fcu - md_out})
    print(f'{nome}: AUC = {a:.4f} · mediana FCU {md_fcu:.4f} × demais {md_out:.4f}')

validacao = pd.DataFrame(val)
print('\ncritério do plano: AUC acima de 0,75 é separação nítida; abaixo de 0,65 é problema sério')
validacao.round(4).to_csv(CAMINHO_FAT / 'nb04_validacao_fcu.csv', sep=';', index=False, encoding='utf-8-sig')
print(validacao.round(4).to_string(index=False))

setores de FCU no conjunto completo: 18,901 · demais: 68,644


índice 0-1: AUC = 0.8131 · mediana FCU 0.4047 × demais 0.3385
escore refinado: AUC = 0.8616 · mediana FCU 0.7289 × demais -0.2523

critério do plano: AUC acima de 0,75 é separação nítida; abaixo de 0,65 é problema sério
         medida    auc  mediana_fcu  mediana_demais  diferenca
     índice 0-1 0.8131       0.4047          0.3385     0.0662
escore refinado 0.8616       0.7289         -0.2523     0.9812


In [15]:
# ── Figura: curva ROC ────────────────────────────────────────────────────────
# FORMA: uma série é o ponto (o índice), a diagonal é só referência — logo *ênfase*, e não
# duas séries categóricas. A curva vai em clay, cheia e grossa; o acaso em cinza,
# tracejado e fino. A AUC entra como número-herói dentro do gráfico, que é a informação
# que o leitor de fato quer, e dispensa ler a curva.
fpr, tpr = curva_roc(base6['indice_01'].to_numpy(), base6['fcu'])
auc = validacao.loc[0, 'auc']

fig, ax = plt.subplots(figsize=(4.8, 4.8), facecolor=SURF)
ax.plot([0, 1], [0, 1], color=CINZA, lw=1.2, ls='--', zorder=1)
ax.plot(fpr, tpr, color=CLAY, lw=2.4, zorder=3)
ax.fill_between(fpr, tpr, alpha=0.08, color=CLAY, zorder=2)
ax.annotate('acaso', (0.62, 0.58), color=CINZA, fontsize=9, rotation=39)
ax.text(0.97, 0.10, f'AUC {auc:.3f}'.replace('.', ','), ha='right', color=CLAY, fontsize=20)
ax.text(0.97, 0.045, f'{n_fcu:,} setores de FCU contra {n_out:,}'.replace(',', '.'),
        ha='right', color=CINZA, fontsize=8.5)
ax.set_xlabel('falso-positivos (1 − especificidade)', color=CINZA, fontsize=9)
ax.set_ylabel('verdadeiro-positivos (sensibilidade)', color=CINZA, fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.tick_params(colors=CINZA, labelsize=9)
ax.xaxis.set_major_formatter(VIRGULA); ax.yaxis.set_major_formatter(VIRGULA)
ax.grid(color=CINZA, alpha=0.15, lw=0.8); ax.set_axisbelow(True)
for lado in ('top', 'right'):
    ax.spines[lado].set_visible(False)
for lado in ('left', 'bottom'):
    ax.spines[lado].set_color(CINZA)
ax.set_facecolor(SURF)
ax.set_title('O índice separa os setores de favela?', color=TINTA, fontsize=11, loc='left')
salvar(fig, 'nb04_roc_fcu.png')

figura: banco_de_dados/eda/fatorial/figuras/nb04_roc_fcu.png


## 10. Síntese, decisões e limitações

In [16]:
# ── Tabela final e o que fica decidido ───────────────────────────────────────
print('PESOS — solução Varimax, 6 componentes (sem o lixo), pesos empíricos\n')
final = pesos.copy()
final['peso_pct'] = 100 * final['peso']
final['dimensao'] = final['dimensao'].map({1: 'Socioeconômica', 2: 'Saneamento'})
print(final[['dimensao', 'carga', 'peso_pct']].round(3).to_string())
print(f"\nrepartição empírica ortogonal : {100*rep6_ort[0]:.1f} / {100*rep6_ort[1]:.1f}")
print(f"repartição empírica oblíqua   : {100*rep6_obl[0]:.1f} / {100*rep6_obl[1]:.1f}")
print(f"referência IVS-BH 2012        : 60,0 / 40,0")
print(f"Φ entre os fatores            : {phi6[0,1]:.3f}")
print(f"\nSpearman índice 0-1 × escore refinado: {rho:.4f}  (critério do plano: > 0,95)")
print(f"AUC contra os setores de FCU        : {auc:.4f}  (critério do plano: > 0,75)")
print(f"renda × cor/raça na matriz fatorada : {par_listwise:.3f}  (limiar do livro: 0,80)")

final.rename_axis('variavel').round(6).to_csv(
    CAMINHO_FAT / 'nb04_sintese_pesos.csv', sep=';', encoding='utf-8-sig')
print('\nnb04_sintese_pesos.csv')

PESOS — solução Varimax, 6 componentes (sem o lixo), pesos empíricos

                          dimensao  carga  peso_pct
Água inadequada         Saneamento  0.903    22.321
Esgoto inadequado       Saneamento  0.679    12.640
Razão de moradores  Socioeconômica  0.532     7.171
Analfabetismo 15+   Socioeconômica  0.868    19.093
Renda (invertida)   Socioeconômica  0.915    21.208
Cor/raça PPI        Socioeconômica  0.833    17.567

repartição empírica ortogonal : 65.0 / 35.0
repartição empírica oblíqua   : 65.8 / 34.2
referência IVS-BH 2012        : 60,0 / 40,0
Φ entre os fatores            : 0.522

Spearman índice 0-1 × escore refinado: 0.9241  (critério do plano: > 0,95)
AUC contra os setores de FCU        : 0.8131  (critério do plano: > 0,75)
renda × cor/raça na matriz fatorada : 0.784  (limiar do livro: 0,80)

nb04_sintese_pesos.csv


### Três resultados que contrariam o que os documentos previam

Registrados aqui porque a diferença entre achado e erro importa, e nos três casos a
verificação apontou para achado.

1. **A concordância entre o índice 0–1 e o escore refinado não atingiu o critério**, e a
   forma como ela falha é mais informativa do que o número. O plano previa Spearman acima
   de 0,97 e fixou 0,95 como validação. Deu **0,924** com o escore calculado sobre os
   postos — que é o coerente, porque a solução fatorial foi estimada sobre a matriz de
   Spearman — e **0,945** com o escore calculado sobre os valores brutos padronizados, que
   é o incoerente. Os dois ficam acima do piso de 0,90 que obrigaria a trocar o índice
   oficial pelo escore refinado, e os dois ficam abaixo de 0,95.

   O sentido da diferença é o ponto: **quanto mais coerente o escore fica com o modelo
   fatorial, mais ele se afasta do índice planejado**. A razão é que o índice 0–1 é
   min-max de valores **brutos** e o modelo fatorial vive nos **postos**. Não é
   instabilidade amostral — o bootstrap mediu a incerteza dos pesos em 0,59 ponto
   percentual, que é desprezível. É a limitação 2 abaixo cobrando o seu preço, e ela não
   se resolve com mais reamostragem: ou o índice passa a ser composto sobre postos, ou a
   fatorial passa a ser estimada sobre Pearson, ou a divergência é declarada. As três são
   defensáveis; escolher sem saber que se está escolhendo, não.
2. **A repartição oblíqua andou para longe da literatura, não para perto.** O plano previa
   que a rotação oblíqua deslocasse os pesos de 65/35 na direção dos 60/40 do IVS-BH. Deu
   65,8/34,2 — meio ponto no sentido oposto. A convergência com a literatura, portanto,
   não depende da escolha de rotação.
3. **Nenhum par da matriz fatorada cruza o limiar de multicolinearidade.** Ver a
   limitação 3 abaixo.

Um quarto ponto, que confirma em vez de contrariar: na solução de sete variáveis, o eixo
principal atribui ao lixo comunalidade de **0,052**, contra 0,859 da ACP. A divergência de
0,836 é a maior da tabela do bloco 4 e diz, com mais força do que a ACP dizia, que o lixo
não tem variância comum com o construto — o fator próprio que ele forma na ACP é variância
específica, que a análise fatorial não conta.

### As decisões que vão para a orientação

Nenhuma delas se fecha aqui. O que este notebook entrega é o custo de cada opção.

| # | Decisão | Evidência produzida |
|---|---|---|
| 1 | Pesos empíricos ou 60/40 da literatura | os dois convergem; a tabela de contingência do bloco 8 diz quantos setores mudam de faixa |
| 2 | Destino do indicador de lixo | forma fator próprio; sem ele a variância acumulada sobe e a estrutura teórica aparece limpa |
| 3 | Política do sigilo no analfabetismo | o cenário sem a variável está no bloco 8, com os setores recuperados e a mudança de faixa |
| 4 | Um fator ou dois | Kaiser e Horn dizem um na solução sem lixo; a teoria diz dois; o custo está no bloco 8 |
| 5 | Rotação ortogonal ou oblíqua | Φ está calculado no bloco 5, e com ele a repartição oblíqua dos pesos |

### Limitações declaradas

1. **O Bartlett é vazio nesta escala.** Com 87.545 casos o teste rejeita H₀ por construção
   (livro, p. 43). A adequabilidade se apoia no KMO e nos MSA.
2. **ACP sobre matriz de Spearman é ACP de postos.** As cargas se referem a posições
   relativas, não a magnitudes, e os escores herdam essa natureza ordinal. A solução de
   Pearson está no bloco 2 como sensibilidade.
3. **Multicolinearidade renda × cor/raça a 0,784** na matriz efetivamente fatorada —
   **abaixo** do limiar de 0,80 da p. 42. O −0,811 que os documentos do projeto citam é a
   correlação par a par da EDA, calculada sobre os 104.108 setores do recorte, não sobre
   os 87.545 da matriz listwise. Os dois números estão certos e medem conjuntos
   diferentes; o artigo precisa citar o da matriz que foi fatorada.
4. **Viés não aleatório do sigilo:** 16.563 setores (15,9%) perdidos por *listwise*, quase
   todos pelo analfabetismo, e o sigilo incide nos setores de melhor situação (§6.2.6 do
   Guia). A amostra é enviesada para os mais vulneráveis. O livro da Enap não trata de
   dados faltantes em nenhuma das 74 páginas.
5. **Dependência espacial não tratada.** A análise fatorial pressupõe unidades
   independentes; setores vizinhos não são. A autocorrelação infla a covariação e, com
   ela, autovalores e cargas. O I de Moran dos escores, na etapa de geoprocessamento, dará
   a medida do problema.
6. **Falácia ecológica.** As unidades são territórios. Toda carga descreve covariação entre
   setores e nada afirma sobre indivíduos.
7. **A padronização usada aqui é min-max global e provisória.** A normalização por
   município é do Notebook 03, e a ordem entre as duas etapas foi medida: fatorar depois de
   normalizar derruba o KMO para 0,720 e a repartição para 56/44.
8. **`renda_media_sem_extremo` não entrou.** A coluna existe no entregável desde a 2ª
   rodada da EDA, e toda a análise fatorial — inclusive os CSVs de referência de agosto —
   usa `renda_media`. Qual das duas vai ao índice é decisão pendente, e refazer a fatorial
   sobre a renda sem extremo é trabalho de uma rodada.
9. **Reflexivo ou formativo.** Se o IVS é um construto que *causa* os indicadores
   (reflexivo, que é o que a análise fatorial pressupõe) ou um índice *composto por* eles
   (formativo, em que a fatorial não seria o instrumento adequado) é questão conceitual em
   aberto, e precisa de literatura própria e decisão da orientação.

### Reprodutibilidade

Matemática em `src/ivs_censo/fatorial.py`, testada em `tests/test_fatorial.py`. Entrada:
o `.db` da entrega. Saídas em `banco_de_dados/eda/fatorial/` com prefixo `nb04_`, e figuras
em `figuras/`. Rodar com:

```
jupyter execute notebooks/Fase3_EDA_ELSI/04_Analise_Fatorial.ipynb
```